# agro_ministry_news pipeline

As of 2026-09-14, the actual scraping/sampling/classification-prompt code lives in
**`agroministrynews_module.py`** (`TarimOrmanScraper`, `ValidationSampleBuilder`) rather than
being reimplemented inline in this notebook — mirrors the pattern `resmi_gazete/`'s
`resmigazete_module.py` uses. This notebook now just shows example usage of that module, step
by step: scrape -> grow the validation sample -> build the classification prompt (handed to a
Claude Code subagent, since there's no separate Anthropic API key — see
`agent_note_agroministrynews_FSOI.md`) -> fold the subagent's output back in.

Three cells that used to live here were deleted (2026-09-14, per Orhan) because they're
dead/superseded, not because the work they represent was worthless — see
`agent_note_agroministrynews_FSOI.md`'s "Notebook cleanup" section for what each one was and
why it's gone: the original title/date-only scraper (`agroforest_ministry_news.xlsx`
pipeline), the early title-only "tohum" keyword filter, and the `zeyrek`-based lemmatization
cell (abandoned when the strand switched to Claude-based classification).

The full-text scraper (`scrape_tarimorman_news_fulltext`, formerly in this cell) has already
been run to completion — the corpus is fully scraped (`agroforestministry_news.csv`, Number
153-7260). The cell below is for reference/reuse (e.g. re-scraping to catch new articles
published since), not something that needs re-running now.

In [ ]:
from agroministrynews_module import TarimOrmanScraper

# Example usage - already run to completion (Number 153-7260), not re-run here.
# scraper = TarimOrmanScraper()
# scraper.scrape_resumable(start_number=153, end_number=7260, output_file="agroforestministry_news.csv")

## Validation sample pipeline

`ValidationSampleBuilder` wraps the grow-sample / classification-prompt / label-append steps
that were previously only documented (not coded) in `agent_note_agroministrynews_FSOI.md`.
As of this notebook revision the sample is at 500 rows — the cells below show how more rows
would be added and classified, not something to run blindly (check the agent note for
current status first). No batch numbering (dropped 2026-09-14, see agent note) — growing the
sample and finding what's unclassified are both just set operations on article `Number`,
nothing needs to be tracked or numbered.

In [ ]:
from agroministrynews_module import ValidationSampleBuilder

builder = ValidationSampleBuilder(
    source_csv="agroforestministry_news.csv",
    sample_csv="agroforestministry_news_validation_sample.csv",
    sample_labels_csv="agroforestministry_news_validation_sample_CLAUDE_LABELS.csv",
)

# Step 1: grow the sample (plain random, excludes Numbers already in sample_csv, ends with a
# mandatory duplicate check - see agent_note, "Duplicate-check rule"). No batch number - these
# rows are just new additions to the pool.
# builder.grow_sample(n=100, seed=53)

# Step 2: find what's unclassified so far (Numbers in the sample with no matching row in the
# sample labels file yet) and write those out for a classification subagent.
# builder.get_unclassified(out_path="to_classify.csv")

In [ ]:
# Step 3: build the exact prompt text to hand to the Agent tool
# (subagent_type="general-purpose", model="sonnet" - no separate Anthropic API key exists,
# classification only happens through an interactive Claude Code session's own subagent
# mechanism, not a script-driven call to Anthropic's API from this notebook).

# prompt_text = builder.build_classification_prompt(
#     n_rows=100,
#     input_csv_path="to_classify.csv",
#     output_csv_path="new_labels.csv",
# )
# print(prompt_text)  # copy this into the Agent tool call

# Step 4: once the subagent has written new_labels.csv, fold it into the sample labels file
# (adds the empty Orhan_Category column automatically).
# builder.append_labels("new_labels.csv")